In [0]:
SILVER_BASE_PATH = "/Volumes/main/default/datalake_retail/silver"
GOLD_BASE_PATH   = "/Volumes/main/default/datalake_retail/gold"

In [0]:
sales_silver = spark.read.format("delta").load(f"{SILVER_BASE_PATH}/sales_transactions")
products_silver = spark.read.format("delta").load(f"{SILVER_BASE_PATH}/products")
stores_silver = spark.read.format("delta").load(f"{SILVER_BASE_PATH}/stores")

In [0]:
#GOLD TABLE 1 – Daily Sales Summary
from pyspark.sql.functions import sum, count

daily_sales_summary = (
    sales_silver
    .groupBy("transaction_date")
    .agg(
        sum("total_amount").alias("daily_revenue"),
        sum("quantity").alias("total_quantity_sold"),
        count("transaction_id").alias("transaction_count")
    )
)

In [0]:
daily_sales_summary.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/daily_sales_summary")

In [0]:
#GOLD TABLE 2 – Monthly Revenue by Region
from pyspark.sql.functions import year, month

monthly_revenue_region = (
    sales_silver
    .join(stores_silver, "store_id", "inner")
    .withColumn("year", year("transaction_date"))
    .withColumn("month", month("transaction_date"))
    .groupBy("year", "month", "region")
    .agg(
        sum("total_amount").alias("monthly_revenue")
    )
)

In [0]:
monthly_revenue_region.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("year", "month") \
    .save(f"{GOLD_BASE_PATH}/monthly_revenue_region")

In [0]:
#GOLD TABLE 3 – Product Performance Metrics
product_performance = (
    sales_silver
    .join(products_silver, "product_id", "inner")
    .groupBy("product_id", "product_name", "category")
    .agg(
        sum("total_amount").alias("total_revenue"),
        sum("quantity").alias("total_units_sold"),
        count("transaction_id").alias("transaction_count")
    )
)

In [0]:
product_performance.write.format("delta") \
    .mode("overwrite") \
    .save(f"{GOLD_BASE_PATH}/product_performance")

In [0]:
%sql
OPTIMIZE delta.`/Volumes/main/default/datalake_retail/gold/daily_sales_summary`;

path,metrics
dbfs:/Volumes/main/default/datalake_retail/gold/daily_sales_summary,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1766433170796, 1766433171580, 8, 0, null, List(0, 0), null, 4, 4, 0, 0, null)"


In [0]:
%sql
OPTIMIZE delta.`/Volumes/main/default/datalake_retail/gold/monthly_revenue_region`;

path,metrics
dbfs:/Volumes/main/default/datalake_retail/gold/monthly_revenue_region,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 12, null, null, 0, 0, 12, 12, true, 0, 0, 1766433182388, 1766433183208, 8, 0, null, List(0, 0), null, 4, 4, 0, 0, null)"


In [0]:
%sql
OPTIMIZE delta.`/Volumes/main/default/datalake_retail/gold/product_performance`;

path,metrics
dbfs:/Volumes/main/default/datalake_retail/gold/product_performance,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1766433188641, 1766433189047, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null)"


In [0]:
spark.read.format("delta").load(f"{GOLD_BASE_PATH}/daily_sales_summary").count()

365

In [0]:
spark.read.format("delta").load(f"{GOLD_BASE_PATH}/monthly_revenue_region").count()

72

In [0]:
spark.read.format("delta").load(f"{GOLD_BASE_PATH}/product_performance").count()

101

In [0]:
dbutils.fs.ls("/Volumes/main/default/datalake_retail/gold")

[FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/gold/daily_sales_summary/', name='daily_sales_summary/', size=0, modificationTime=1766433234963),
 FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/gold/monthly_revenue_region/', name='monthly_revenue_region/', size=0, modificationTime=1766433234963),
 FileInfo(path='dbfs:/Volumes/main/default/datalake_retail/gold/product_performance/', name='product_performance/', size=0, modificationTime=1766433234963)]

In [0]:
%sql
CREATE OR REPLACE TABLE main.default.daily_sales_summary
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/main/default/datalake_retail/gold/daily_sales_summary`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.default.monthly_revenue_region
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/main/default/datalake_retail/gold/monthly_revenue_region`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE main.default.product_performance
USING DELTA
AS
SELECT *
FROM delta.`/Volumes/main/default/datalake_retail/gold/product_performance`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT COUNT(*)
FROM delta.`/Volumes/main/default/datalake_retail/gold/daily_sales_summary`;

COUNT(*)
365


In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default
